# Provider evaluation: fill rate, freshness, comparison, ranking

**Input:** the next cell accepts `Archive.zip` (or another zip path / Colab upload). It cleans the working folder, unzips the archive, and removes junk files (`__MACOSX`, `._*`). Provider subfolders (`people_context/`, `the_swarm/`, …) must be at top level inside the zip. Then the notebook discovers structure, runs LLM field mapping when needed, and computes fill rate, freshness, comparison, scoring, and recommendation.

Add `OPENROUTER_API_KEY` in Colab Secrets (or env) for LLM calls (field mapping and final report).

In [ ]:
# Setup: install deps, load API key (Colab Secrets)
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q pandas openpyxl requests json5

import os
import json
import json5
import re
from pathlib import Path
from datetime import datetime
import pandas as pd
import requests
try:
    from IPython.display import display
except ImportError:
    display = print

def parse_llm_json(content: str, expect_array: bool = False):
    """Extract and parse JSON from LLM response (handles markdown fences and trailing commas via json5)."""
    text = content.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```\s*$", "", text)
    start = text.find("[") if expect_array else text.find("{")
    if start < 0:
        start = text.find("{") if not expect_array else text.find("[")
    if start < 0:
        raise ValueError("No JSON object or array found")
    open_b, close_b = ("[", "]") if text[start] == "[" else ("{", "}")
    depth = 0
    for i in range(start, len(text)):
        if text[i] == open_b: depth += 1
        elif text[i] == close_b:
            depth -= 1
            if depth == 0:
                return json5.loads(text[start : i + 1])
    raise ValueError("Unbalanced brackets")

# OpenRouter API key: in Colab add Secret OPENROUTER_API_KEY
def get_api_key():
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get("OPENROUTER_API_KEY")
        except Exception as e:
            print("Add OPENROUTER_API_KEY in Colab Secrets (key icon in left sidebar).", e)
            return None
    return os.environ.get("OPENROUTER_API_KEY")

OPENROUTER_API_KEY = get_api_key()
OPENROUTER_MODEL = "anthropic/claude-sonnet-4.6"
print("OpenRouter key:", "set" if OPENROUTER_API_KEY else "NOT SET")
print("OpenRouter model:", OPENROUTER_MODEL)

# Root folder: cleaned and filled by the next cell (zip upload)
ROOT_DATA = Path("/content") if IN_COLAB else Path(os.getcwd())

# Logging: capture all messages in memory, write to file at the end of the run
import logging
LOG_FILE = ROOT_DATA / "provider_evaluation_run.log"
LOG_LINES = []  # in-memory buffer for full run

class MemoryHandler(logging.Handler):
    def emit(self, record):
        LOG_LINES.append(self.format(record))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        MemoryHandler(),
        logging.StreamHandler(),
    ],
    force=True,
)
log = logging.getLogger("provider_eval")
log.info("Setup done. ROOT_DATA=%s, OPENROUTER_API_KEY=%s, OPENROUTER_MODEL=%s", ROOT_DATA, "set" if OPENROUTER_API_KEY else "NOT SET", OPENROUTER_MODEL)
print("Log: in-memory (will write to file at end)")

## Input: Archive.zip (or upload) – clean workspace and unzip

**Input:** path to a zip file (default `Archive.zip` in project dir or `/content`). In Colab you can upload a zip instead. The cell (1) cleans the working folder (removes old provider data; keeps `output/`), (2) unzips your file, (3) removes junk (`__MACOSX`, `._*` files). If the zip has a single root folder, its contents are moved to the root.

In [ ]:
import zipfile
import shutil
import tempfile

# Input: zip file name (in ROOT_DATA or cwd), or Colab upload, or env PROVIDER_ZIP
INPUT_ZIP = "Archive.zip"

# Folders to keep when cleaning (do not delete)
KEEP_DIRS = {"output"}

def clean_workspace(root: Path) -> None:
    """Remove provider data dirs and files under root; keep KEEP_DIRS."""
    for item in root.iterdir():
        if item.name.startswith("."):
            continue
        if item.is_dir():
            if item.name in KEEP_DIRS:
                continue
            shutil.rmtree(item)
            print("Removed dir:", item.name)
        else:
            item.unlink()
            print("Removed file:", item.name)

def remove_junk_files(root: Path) -> None:
    """Remove __MACOSX and all ._* files/dirs (macOS metadata) under root."""
    macosx = root / "__MACOSX"
    if macosx.exists():
        shutil.rmtree(macosx)
        print("Removed __MACOSX")
    removed = 0
    for path in sorted(root.rglob(".*"), key=lambda p: -len(p.parts)):
        if path.name.startswith("._"):
            if path.is_file():
                path.unlink()
                removed += 1
            elif path.is_dir() and not any(path.iterdir()):
                path.rmdir()
                removed += 1
    if removed:
        print("Removed", removed, "junk files (._*)")

def get_zip_path() -> str:
    """Return path to the zip: Colab upload, or INPUT_ZIP / PROVIDER_ZIP in ROOT_DATA or cwd."""
    if IN_COLAB:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded. Run the cell and choose a zip (e.g. Archive.zip).")
        name = list(uploaded.keys())[0]
        if not name.lower().endswith(".zip"):
            raise RuntimeError("Upload a .zip file.")
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".zip")
        tmp.write(uploaded[name])
        tmp.close()
        return tmp.name
    zip_name = os.environ.get("PROVIDER_ZIP", INPUT_ZIP)
    path = Path(zip_name)
    if not path.is_absolute():
        for base in (ROOT_DATA, Path.cwd()):
            candidate = base / zip_name
            if candidate.exists():
                return str(candidate)
        raise FileNotFoundError(f"ZIP not found: {zip_name}. Place {INPUT_ZIP} in project dir or set PROVIDER_ZIP.")
    if not path.exists():
        raise FileNotFoundError(f"ZIP not found: {path}")
    return str(path)

def unzip_and_flatten(zip_path: str, root: Path) -> None:
    """Extract zip to root. If single top-level dir (not in KEEP_DIRS), move its contents up."""
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(root)
    subdirs = [d for d in root.iterdir() if d.is_dir() and d.name not in KEEP_DIRS and not d.name.startswith(".")]
    if len(subdirs) == 1:
        single = subdirs[0]
        for child in single.iterdir():
            dest = root / child.name
            if dest.exists():
                shutil.rmtree(dest) if dest.is_dir() else dest.unlink()
            shutil.move(str(child), str(root))
        single.rmdir()
        print("Flattened single root folder:", single.name)

# 1) Clean workspace (keep output only)
print("Cleaning workspace...")
clean_workspace(ROOT_DATA)
# 2) Resolve and unzip input (Archive.zip or upload)
zip_path = get_zip_path()
print("Extracting:", zip_path)
unzip_and_flatten(zip_path, ROOT_DATA)
# 3) Remove junk: __MACOSX and ._* files
remove_junk_files(ROOT_DATA)
folders = [p.name for p in ROOT_DATA.iterdir() if p.is_dir() and not p.name.startswith(".")]
print("Done. Provider folders:", folders)
log = logging.getLogger("provider_eval")
log.info("Input: zip_path=%s, extracted_to=%s, folders=%s", zip_path, ROOT_DATA, folders)

In [ ]:
# Config: ROOT_DATA set in Setup, filled by zip upload cell; scoring weights; stale threshold (days)
# No caching: structures and mappings are computed in memory each run.

# Scoring weights (fill_rate, freshness, volume). Freshness: lower days_ago = better.
FILL_RATE_WEIGHT = 0.4
FRESHNESS_WEIGHT = 0.3
VOLUME_WEIGHT = 0.3
STALE_DAYS = 365  # profile updated more than this = stale

# Key fields for comparison and scoring; set by LLM from data preview (see cell after Build structure), else default
KEY_FIELDS = [
    "email", "phone", "current_job_title", "current_company", "skills_keywords",
    "work_history", "education", "location", "linkedin_url"
]
dirs_list = [p.name for p in ROOT_DATA.iterdir() if p.is_dir() and not p.name.startswith(".")][:10]
print("ROOT_DATA:", ROOT_DATA)
print("Provider subfolders expected under ROOT_DATA:", dirs_list)
log = logging.getLogger("provider_eval")
log.info("Config: ROOT_DATA=%s, dirs=%s", ROOT_DATA, dirs_list)

## 1. Scan provider folders (file list + preview)

In [ ]:
def get_preview(path: Path, max_chars: int = 4000) -> dict:
    """Return type and short preview for a file (keys for JSON, header for CSV, sheet names for Excel)."""
    suf = path.suffix.lower()
    try:
        if suf == ".json":
            with open(path, encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, list) and data:
                return {"type": "json_array", "keys": list(data[0].keys()) if isinstance(data[0], dict) else []}
            if isinstance(data, dict):
                return {"type": "json_object", "keys": list(data.keys())}
            return {"type": "json"}
        if suf == ".csv":
            df = pd.read_csv(path, nrows=2)
            return {"type": "csv", "columns": list(df.columns), "rows": df.head(2).to_dict("records")}
        if suf in (".xlsx", ".xls"):
            xl = pd.ExcelFile(path)
            df = pd.read_excel(path, sheet_name=xl.sheet_names[0], nrows=3)
            return {"type": "excel", "sheets": xl.sheet_names, "columns": list(df.columns), "sample_rows": df.head(2).to_dict("records")}
    except Exception as e:
        return {"type": suf[1:], "error": str(e)}
    return {"type": "unknown"}

# Dirs under ROOT_DATA that are not data providers (skip when scanning)
SKIP_PROVIDER_SCAN = {"provider_structures", "provider_mappings", "output", "__MACOSX"}

def scan_providers(root: Path) -> dict:
    """Scan root for provider subdirs; return {provider_name: [{path, preview}, ...]}."""
    result = {}
    for item in root.iterdir():
        if not item.is_dir() or item.name.startswith(".") or item.name in SKIP_PROVIDER_SCAN:
            continue
        files = []
        for f in item.rglob("*"):
            if f.is_file() and f.suffix.lower() in (".json", ".csv", ".xlsx", ".xls"):
                rel = f.relative_to(item)
                files.append({"path": str(rel), "full_path": str(f), "preview": get_preview(f)})
        if files:
            result[item.name] = files
    return result

scanned = scan_providers(ROOT_DATA)
if not scanned:
    all_dirs = [p.name for p in ROOT_DATA.iterdir() if p.is_dir() and not p.name.startswith(".")]
    print("No provider folders with data files found.")
    print("Dirs under ROOT_DATA:", all_dirs)
    print("Skipped (not providers):", SKIP_PROVIDER_SCAN)
    print("Each provider dir must contain at least one .json, .csv, or .xlsx/.xls file.")
else:
    print("Providers and file counts:")
    log = logging.getLogger("provider_eval")
    for prov, flist in scanned.items():
        print(f"  {prov}: {len(flist)} files")
        log.info("Scan provider %s: %d files, paths=%s", prov, len(flist), [f["path"] for f in flist[:10]])
        for f in flist[:5]:
            print(f"    - {f['path']} -> {f['preview'].get('type', '?')}")

## 2. LLM: which file is samples, which is volumes (provider structure)

In [ ]:
def openrouter_chat(messages: list, api_key: str = None) -> str:
    """Call OpenRouter chat completions; return assistant content."""
    key = api_key or OPENROUTER_API_KEY
    if not key:
        return ""
    r = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
        json={"model": OPENROUTER_MODEL, "messages": messages},
        timeout=120,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

# Prepend to messages when expecting a JSON-only response
SYSTEM_JSON = {"role": "system", "content": "You must respond with ONLY a valid JSON object or array. No explanation, no markdown, no text before or after the JSON."}

def ensure_provider_structure(provider_name: str, files_with_preview: list, root: Path) -> dict:
    """Compute provider structure via LLM or heuristics (no cache). Return structure dict."""
    log = logging.getLogger("provider_eval")
    if not OPENROUTER_API_KEY:
        print("No API key; using heuristics for structure.")
        sample_paths = [f["path"] for f in files_with_preview if f["path"].endswith(".json")]
        volume_paths = [f["path"] for f in files_with_preview if f["path"].endswith(".csv") or f["path"].endswith(".xlsx")]
        has_query_in_profile = any("_query" in (f.get("preview") or {}).get("keys", []) for f in files_with_preview if f["path"].endswith(".json"))
        structure = {
            "sample_files": sample_paths[:1] if len(sample_paths) == 1 else sample_paths,
            "volume_file": volume_paths[0] if volume_paths else "",
            "volume_format": {"query_col": "query", "region_col": "region", "total_col": "total"},
            "query_region_in_samples": "from_profile" if has_query_in_profile else "from_filename",
        }
        return structure
    print(f"  {provider_name}: asking LLM for structure...")
    prompt = f"""Provider folder: {provider_name}. Files (path -> preview):
{json.dumps([{"path": f["path"], "preview": f["preview"]} for f in files_with_preview], indent=2)}

Return a single JSON object (no markdown) with:
- "sample_files": list of paths that contain profile samples (array of person records). If multiple files, say how to get query/region from filename (e.g. "role_region").
- "volume_file": path to file with total counts per query/region (or "" if none).
- "volume_format": {{ "query_col": "...", "region_col": "...", "total_col": "..." }} for that file (column names or sheet/column for Excel).
- "query_region_in_samples": "from_profile" if each record has query/region fields, or "from_filename" and describe pattern (e.g. "filename without ext = role_region").
"""
    content = openrouter_chat([SYSTEM_JSON, {"role": "user", "content": prompt}])
    try:
        structure = parse_llm_json(content)
    except Exception as e:
        print("LLM structure parse failed:", e)
        structure = {"sample_files": [f["path"] for f in files_with_preview if f["path"].endswith(".json")], "volume_file": "", "volume_format": {}, "query_region_in_samples": "from_filename"}
    log.info("Structure %s: LLM computed, keys=%s, volume_format=%s", provider_name, list(structure.keys()), structure.get("volume_format"))
    print(f"  {provider_name}: structure ready")
    return structure

def _load_first_sample_for_rule(structure: dict, root: Path) -> dict:
    """Load first profile from first sample file for LLM context."""
    sample_paths = structure.get("sample_files") or []
    if isinstance(sample_paths, str):
        sample_paths = [sample_paths]
    for rel in sample_paths:
        path = root / rel
        if path.exists() and path.suffix.lower() == ".json":
            with open(path, encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, list) and data:
                return data[0]
            if isinstance(data, dict):
                return data
    return {}

def ensure_query_region_rule(provider_name: str, structure: dict, root: Path, files_with_preview: list) -> dict:
    """If query_region_rule missing in structure, call LLM to get rule from sample paths and optional first record. Store in structure (no cache)."""
    log = logging.getLogger("provider_eval")
    if structure.get("query_region_rule"):
        log.info("Query/region rule %s: already in structure", provider_name)
        return structure
    sample_paths = structure.get("sample_files") or []
    if isinstance(sample_paths, str):
        sample_paths = [sample_paths]
    if not sample_paths:
        return structure
    from_filename = (structure.get("query_region_in_samples") or "").lower() == "from_filename"
    if not OPENROUTER_API_KEY:
        rule = {"source": "from_filename" if from_filename else "from_profile", "fallback": "heuristic"}
        if from_filename:
            rule["stem_separator"] = "_"
            rule["region_from_last_token"] = True
            rule["region_mapping"] = {"a": "LATAM", "b": "NA_EU_APAC"}
            rule["query_mapping"] = {}
        else:
            rule["query_path"] = "_query"
            rule["region_path"] = "_region"
        structure["query_region_rule"] = rule
        log.info("Query/region rule %s: no API key, heuristic rule", provider_name)
        return structure
    first_sample = _load_first_sample_for_rule(structure, root)
    sample_preview = json.dumps(first_sample, ensure_ascii=False)[:3000] if first_sample else ""
    # Include volume file context to help LLM resolve filename tokens (e.g. 'a'/'b') to real region names
    vf_name = structure.get("volume_file") or ""
    volume_ctx = ""
    if vf_name:
        for _f in files_with_preview:
            if _f["path"] == vf_name:
                _prev = _f.get("preview") or {}
                _rows = _prev.get("sample_rows") or []
                if _rows:
                    volume_ctx = f"\nVolume file (use to resolve region labels from filename tokens):\nColumns: {json.dumps(_prev.get('columns', []))}\nSample rows: {json.dumps(_rows[:4], default=str)}"
                break
    prompt = f"""Sample file paths for this provider: {json.dumps(sample_paths)}
Query/region in samples: {structure.get('query_region_in_samples', '')}
First profile record (optional):
{sample_preview}{volume_ctx}

Determine how to extract (query, region) for each profile. Return a JSON object with:
- "source": "from_filename" or "from_profile"
- If from_profile: "query_path" and "region_path" (dot paths or top-level keys, e.g. "_query", "_region")
- If from_filename: "stem_separator" (e.g. "_"), "region_from_last_token": true, "region_mapping" (map each filename token to canonical region name, e.g. {{"a": "LATAM", "b": "NA_EU_APAC"}} - infer real names from the volume file rows above), "query_mapping" (map filename stem to canonical query name, e.g. {{"software_engineer_react": "SWE_React"}})
Use exact keys. For from_filename: query = stem before last token, region = last token mapped via region_mapping."""
    log.info("Query/region rule %s: calling LLM, sample_paths_len=%d", provider_name, len(sample_paths))
    content = openrouter_chat([SYSTEM_JSON, {"role": "user", "content": prompt}])
    log.info("Query/region rule %s: LLM response len=%d, preview=%s", provider_name, len(content), (content[:200] + "...") if len(content) > 200 else content)
    try:
        rule = parse_llm_json(content)
    except Exception as e:
        log.warning("Query/region rule %s: parse failed %s, using heuristic", provider_name, e)
        rule = {"source": "from_filename" if from_filename else "from_profile", "fallback": "heuristic"}
        if from_filename:
            rule["stem_separator"] = "_"
            rule["region_from_last_token"] = True
            rule["region_mapping"] = {"a": "LATAM", "b": "NA_EU_APAC"}
            rule["query_mapping"] = {}
        else:
            rule["query_path"] = "_query"
            rule["region_path"] = "_region"
    structure["query_region_rule"] = rule
    log.info("Query/region rule %s: rule source=%s", provider_name, rule.get("source"))
    return structure



def ensure_volume_columns(provider_name: str, structure: dict, root: Path) -> dict:
    """Resolve volume file column mapping by showing LLM the actual column names from the file.
    LLM picks from the real columns, so the result is always valid. Updates structure['volume_format']."""
    log = logging.getLogger("provider_eval")
    vf = structure.get("volume_file") or ""
    if not vf:
        return structure
    path = root / vf
    if not path.exists():
        return structure
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path, nrows=5)
    else:
        df = pd.read_excel(path, sheet_name=0, nrows=5)
    actual_cols = list(df.columns)
    sample_rows = df.dropna(how="all").head(3).to_dict("records")
    if not OPENROUTER_API_KEY:
        log.info("Volume columns %s: no API key, keeping existing volume_format", provider_name)
        return structure
    prompt = f"""Volume file for provider "{provider_name}".
Actual columns: {json.dumps(actual_cols)}
Sample rows (first 3 non-empty):
{json.dumps(sample_rows, default=str, indent=2)}

Identify which column serves each role:
- "query_col": column with the search query / role / job title being searched
- "region_col": column with geographic region or search scope (e.g. LATAM, country, query group)
- "total_col": column with a numeric count of results or profiles

Return a JSON object with exactly these three keys. Values must be exact column names from the list above, or "" if not present."""
    log.info("Volume columns %s: calling LLM, actual_cols=%s", provider_name, actual_cols)
    content = openrouter_chat([SYSTEM_JSON, {"role": "user", "content": prompt}])
    log.info("Volume columns %s: LLM response=%s", provider_name, content[:300] if len(content) > 300 else content)
    try:
        resolved = parse_llm_json(content)
        for key in ["query_col", "region_col", "total_col"]:
            val = resolved.get(key, "")
            if val and val not in actual_cols:
                log.warning("Volume columns %s: LLM returned unknown column %s=%r, clearing", provider_name, key, val)
                resolved[key] = ""
        structure["volume_format"] = resolved
        log.info("Volume columns %s: resolved volume_format=%s", provider_name, resolved)
    except Exception as e:
        log.warning("Volume columns %s: LLM parse failed %s, keeping existing volume_format", provider_name, e)
    return structure

print("OK: ensure_provider_structure, ensure_query_region_rule, ensure_volume_columns defined. Run the next cell to build provider structures.")

In [ ]:
# Build structure for each provider (LLM or heuristics, no cache)
provider_structures = {}
try:
    _scanned = scanned
except NameError:
    _scanned = {}
    print("scanned is not defined. Run in order: 1) Upload ZIP, 2) Config, 3) Scan provider folders, then this cell.")

print("Step: building provider structures...")
if not _scanned:
    print("No providers to process. Run the 'Scan provider folders' cell first (section 1).")
else:
    print("Provider structures (from cache or LLM):")
    log = logging.getLogger("provider_eval")
    for prov, flist in _scanned.items():
        provider_structures[prov] = ensure_provider_structure(prov, flist, ROOT_DATA / prov)
        provider_structures[prov] = ensure_query_region_rule(prov, provider_structures[prov], ROOT_DATA / prov, flist)
        provider_structures[prov] = ensure_volume_columns(prov, provider_structures[prov], ROOT_DATA / prov)
        keys = list(provider_structures[prov].keys())
        print(f"  {prov} -> keys: {keys}")
        log.info("Structure %s: keys=%s, sample_files=%s, volume_file=%s, has_query_region_rule=%s", prov, keys, provider_structures[prov].get("sample_files"), provider_structures[prov].get("volume_file"), bool(provider_structures[prov].get("query_region_rule")))

# Determine KEY_FIELDS via LLM from data preview (no hardcoding)
if OPENROUTER_API_KEY and provider_structures:
    all_keys = set()
    for prov, struct in provider_structures.items():
        root = ROOT_DATA / prov
        sample = _load_first_sample_for_rule(struct, root)
        if isinstance(sample, dict):
            all_keys.update(sample.keys())
    keys_preview = sorted(all_keys)[:80]
    log.info("KEY_FIELDS LLM: collected %d unique keys from preview", len(all_keys))
    prompt = f"""We have profile data with these keys (possibly nested): {json.dumps(keys_preview)}
Return a JSON array of field names (snake_case) that should be used as key for comparing people/data providers (e.g. email, phone, current_job_title, current_company, skills_keywords, work_history, education, location, linkedin_url). Pick the most important 8-12. No markdown, only JSON array."""
    content = openrouter_chat([SYSTEM_JSON, {"role": "user", "content": prompt}])
    log.info("KEY_FIELDS LLM: response len=%d, preview=%s", len(content), (content[:150] + "...") if len(content) > 150 else content)
    try:
        KEY_FIELDS = parse_llm_json(content, expect_array=True)
        if isinstance(KEY_FIELDS, list) and KEY_FIELDS:
            log.info("KEY_FIELDS set via LLM: %s", KEY_FIELDS)
            print("KEY_FIELDS (from LLM):", KEY_FIELDS)
        else:
            log.warning("KEY_FIELDS LLM returned empty or invalid, keeping default")
    except Exception as e:
        log.warning("KEY_FIELDS LLM parse failed: %s, keeping default", e)
else:
    log.info("KEY_FIELDS: using default (no API key or no structures)")
print("Done.")

## 3. LLM: field mapping (key + additional fields)

In [ ]:
def get_first_sample(provider_name: str, structure: dict, root: Path) -> dict:
    """Load first profile from sample files for this provider."""
    sample_paths = structure.get("sample_files") or []
    if isinstance(sample_paths, str):
        sample_paths = [sample_paths]
    for rel in sample_paths:
        path = root / rel
        if not path.exists():
            continue
        if path.suffix.lower() == ".json":
            with open(path, encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, list) and data:
                return data[0]
            if isinstance(data, dict):
                return data
    return {}

def ensure_provider_mapping(provider_name: str, sample: dict, root: Path) -> dict:
    """Compute field mapping via LLM (no cache). Return mapping dict."""
    log = logging.getLogger("provider_eval")
    key_list = ", ".join(KEY_FIELDS)
    sample_str = json.dumps(sample, ensure_ascii=False)[:6000]
    if not OPENROUTER_API_KEY:
        print("No API key; skip mapping (will fail later).")
        return {}
    prompt = f"""Profile sample (JSON):
{sample_str}

We need a field mapping (dot-path into this JSON) for:
1) Key fields (for comparison): {key_list}. Also "updated_at", "query", "region" if inside profile.
2) Any other fields present (e.g. headline, summary, certifications, languages, connections, followers, image_url). For each give the JSON path.

Return a single JSON object: keys = our field names, values = path strings (e.g. "profile.headline", "experience.0.title"). Empty string if field not in data. No markdown, only JSON.
"""
    log.info("Mapping %s: calling LLM, key_fields=%s", provider_name, KEY_FIELDS[:5] if len(KEY_FIELDS) > 5 else KEY_FIELDS)
    content = openrouter_chat([SYSTEM_JSON, {"role": "user", "content": prompt}])
    log.info("Mapping %s: LLM response len=%d, preview=%s", provider_name, len(content), (content[:200] + "...") if len(content) > 200 else content)
    try:
        mapping = parse_llm_json(content)
    except Exception as e:
        log.warning("Mapping %s: LLM parse failed %s", provider_name, e)
        print("LLM mapping parse failed:", e)
        mapping = {}
    return mapping

In [ ]:
# Build mapping for each provider (LLM, no cache)
provider_mappings = {}
log = logging.getLogger("provider_eval")
for prov in scanned:
    root = ROOT_DATA / prov
    struct = provider_structures.get(prov, {})
    first = get_first_sample(prov, struct, root)
    provider_mappings[prov] = ensure_provider_mapping(prov, first, root)
    n = len(provider_mappings[prov])
    print(prov, "->", n, "mapped fields")
    log.info("Mapping %s: %d fields", prov, n)

## 4. Universal interpreter (get value by path; is filled?)

In [ ]:
def get_by_path(obj, path: str):
    """Get value at dot path; support .0. for first array element."""
    if not path or not path.strip():
        return None
    parts = path.replace(".", " ").split()
    cur = obj
    for i, p in enumerate(parts):
        if cur is None: return None
        if p.isdigit():
            idx = int(p)
            if isinstance(cur, list) and 0 <= idx < len(cur):
                cur = cur[idx]
            else:
                return None
        else:
            cur = cur.get(p) if isinstance(cur, dict) else None
    return cur

def is_filled(val) -> bool:
    if val is None: return False
    if isinstance(val, str): return bool(val.strip())
    if isinstance(val, list): return len(val) > 0
    if isinstance(val, dict): return len(val) > 0
    return True

def profile_field_values(profile: dict, mapping: dict) -> dict:
    """Return dict of field_name -> value for all keys in mapping."""
    out = {}
    for field, path in mapping.items():
        out[field] = get_by_path(profile, path)
    return out

## 5. Load all providers (profiles + volumes)

Volume files: read directly from xlsx/csv using exact column names from `volume_format` returned by the LLM (`query_col`, `region_col`, `total_col`). No alias guessing. For wide-format files (no region column), remaining numeric columns are melted into rows.

In [ ]:
def parse_query_region_from_filename(filename: str) -> tuple:
    """Heuristic: extract (query, region) from filename when no LLM rule. Last token a/b -> LATAM/NA_EU_APAC; rest = query (no hardcoded mapping)."""
    base = Path(filename).stem
    parts = base.split("_")
    if len(parts) >= 2 and parts[-1] in ("a", "b"):
        region = "LATAM" if parts[-1] == "a" else "NA_EU_APAC"
        query = "_".join(parts[:-1])
        return (query, region)
    return (base, "")

def apply_query_region_rule(rel_path: str, profile: dict, structure: dict) -> tuple:
    """Return (query, region) using structure["query_region_rule"] if present, else heuristic for from_filename."""
    rule = structure.get("query_region_rule") or {}
    source = (rule.get("source") or "").lower()
    if source == "from_profile" and profile:
        qp = rule.get("query_path", "_query")
        rp = rule.get("region_path", "_region")
        q = profile.get(qp) if qp in profile else (get_by_path(profile, qp) if "." in qp or qp.isdigit() else None)
        r = profile.get(rp) if rp in profile else (get_by_path(profile, rp) if "." in rp or rp.isdigit() else None)
        if q is None and "." in qp: q = get_by_path(profile, qp)
        if r is None and "." in rp: r = get_by_path(profile, rp)
        return (str(q).strip() if q is not None else "", str(r).strip() if r is not None else "")
    if source == "from_filename":
        stem = Path(rel_path).stem
        sep = rule.get("stem_separator", "_")
        parts = stem.split(sep)
        region = ""
        query = stem
        if rule.get("region_from_last_token") and parts:
            last = parts[-1]
            region = rule.get("region_mapping") or {}
            region = region.get(last, last)
            query = "_".join(parts[:-1]) if len(parts) > 1 else stem
        qmap = rule.get("query_mapping") or {}
        query = qmap.get(query, query)
        return (str(query), str(region))
    from_filename = (structure.get("query_region_in_samples") or "").lower() == "from_filename"
    if from_filename:
        return parse_query_region_from_filename(rel_path)
    return (None, None)

def load_profiles(provider_name: str, structure: dict, root: Path) -> list:
    """Load all sample files into one list of profiles; add _query, _region from structure rule or heuristic."""
    sample_paths = structure.get("sample_files") or []
    if isinstance(sample_paths, str):
        sample_paths = [sample_paths]
    from_filename = (structure.get("query_region_in_samples") or "").lower() == "from_filename"
    profiles = []
    for rel in sample_paths:
        path = root / rel
        if not path.exists():
            continue
        if path.suffix.lower() != ".json":
            continue
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        items = data if isinstance(data, list) else [data]
        for p in items:
            if isinstance(p, dict):
                q, r = apply_query_region_rule(rel, p, structure)
                if q is not None: p["_query"] = p.get("_query", q)
                if r is not None: p["_region"] = p.get("_region", r)
                profiles.append(p)
    return profiles

def load_volumes(provider_name: str, structure: dict, root: Path) -> pd.DataFrame:
    """Load volume data using verified column names from structure volume_format (set by ensure_volume_columns)."""
    log = logging.getLogger("provider_eval")
    vf = structure.get("volume_file") or ""
    if not vf:
        return pd.DataFrame(columns=["query", "region", "total"])
    path = root / vf
    if not path.exists():
        log.warning("Volume %s: file not found: %s", provider_name, path)
        return pd.DataFrame(columns=["query", "region", "total"])
    fmt = structure.get("volume_format") or {}
    qc = fmt.get("query_col", "")
    rc = fmt.get("region_col", "")
    tc = fmt.get("total_col", "")
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        df = pd.read_excel(path, sheet_name=0)
    rename = {}
    if qc and qc in df.columns:
        rename[qc] = "query"
    if rc and rc in df.columns:
        rename[rc] = "region"
    if tc and tc in df.columns:
        rename[tc] = "total"
    if rename:
        df = df.rename(columns=rename)
    # Wide format: query present but no region col - remaining numeric cols are regions
    if "query" in df.columns and "region" not in df.columns:
        region_cols = [c for c in df.columns if c != "query" and c != "total" and pd.api.types.is_numeric_dtype(df[c])]
        if region_cols:
            df = df.melt(id_vars=["query"], value_vars=region_cols, var_name="region", value_name="total")
    for c in ["query", "region", "total"]:
        if c not in df.columns:
            df[c] = None
    df = df[["query", "region", "total"]].copy()
    df["total"] = pd.to_numeric(df["total"], errors="coerce")
    df = df.dropna(subset=["query"])
    df = df[df["query"].astype(str).str.strip() != ""]
    return df.reset_index(drop=True)

In [ ]:
# Load profiles and volumes for each provider
data = {}
log = logging.getLogger("provider_eval")
for prov in scanned:
    root = ROOT_DATA / prov
    struct = provider_structures.get(prov, {})
    try:
        profiles = load_profiles(prov, struct, root)
        volumes = load_volumes(prov, struct, root)
    except Exception as e:
        log.exception("Load data %s failed: path=%s, error=%s", prov, root, e)
        profiles, volumes = [], pd.DataFrame()
    mapping = provider_mappings.get(prov, {})
    data[prov] = {"profiles": profiles, "volumes": volumes, "mapping": mapping}
    print(f"{prov}: {len(profiles)} profiles, {len(volumes)} volume rows")
    log.info("Load data %s: profiles=%d, volume_rows=%d", prov, len(profiles), len(volumes))
    if not volumes.empty:
        log.info("Load data %s: volume sample: %s", prov, volumes[["query", "region", "total"]].head(5).to_dict("records"))

## 6. Fill rate (key fields + additional fields)

In [ ]:
def compute_fill_rates(data: dict) -> tuple:
    """Return (key_fields_df, additional_fields_df)."""
    key_rows = []
    add_rows = []
    for prov, d in data.items():
        profiles = d["profiles"]
        mapping = d["mapping"]
        if not profiles or not mapping:
            continue
        n = len(profiles)
        key_vals = {f: 0 for f in KEY_FIELDS}
        add_vals = {}
        for p in profiles:
            vals = profile_field_values(p, mapping)
            for f in KEY_FIELDS:
                if is_filled(vals.get(f)):
                    key_vals[f] += 1
            for f, v in vals.items():
                if f in KEY_FIELDS or f in ("updated_at", "query", "region"):
                    continue
                add_vals[f] = add_vals.get(f, 0) + (1 if is_filled(v) else 0)
        key_rows.append({"provider": prov, **{f: round(100 * key_vals[f] / n, 1) for f in KEY_FIELDS}})
        add_rows.append({"provider": prov, **{f: round(100 * add_vals[f] / n, 1) for f in add_vals}})
    key_df = pd.DataFrame(key_rows) if key_rows else pd.DataFrame()
    add_df = pd.DataFrame(add_rows) if add_rows else pd.DataFrame()
    return key_df, add_df

fill_key, fill_add = compute_fill_rates(data)
log = logging.getLogger("provider_eval")
log.info("Fill rate: KEY_FIELDS=%s, fill_key shape=%s", KEY_FIELDS, fill_key.shape if not fill_key.empty else "empty")
print("Fill rate (key fields for comparison):")
display(fill_key)
if not fill_add.empty:
    print("Fill rate (additional fields):")
    display(fill_add)

## 7. Freshness (days since update; % stale)

In [ ]:
def parse_date(s):
    if not s or not isinstance(s, str):
        return None
    s = s.strip()[:10]
    for fmt in ("%Y-%m-%d", "%d.%m.%Y", "%Y/%m/%d"):
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            continue
    return None

def freshness_stats(data: dict, stale_days: int = 365) -> pd.DataFrame:
    rows = []
    for prov, d in data.items():
        profiles = d["profiles"]
        mapping = d["mapping"]
        if not profiles or "updated_at" not in mapping:
            rows.append({"provider": prov, "median_days_ago": None, "pct_stale_over_1y": None})
            continue
        days_list = []
        for p in profiles:
            val = get_by_path(p, mapping.get("updated_at", ""))
            dt = parse_date(val)
            if dt:
                days_list.append((datetime.now() - dt).days)
        if not days_list:
            rows.append({"provider": prov, "median_days_ago": None, "pct_stale_over_1y": None})
            continue
        med = int(pd.Series(days_list).median())
        stale_pct = round(100 * sum(1 for x in days_list if x > stale_days) / len(days_list), 1)
        rows.append({"provider": prov, "median_days_ago": med, "pct_stale_over_1y": stale_pct})
    return pd.DataFrame(rows)

fresh_df = freshness_stats(data, STALE_DAYS)
log = logging.getLogger("provider_eval")
log.info("Freshness: fresh_df shape=%s, per_provider=%s", fresh_df.shape, fresh_df.to_dict("records") if not fresh_df.empty else [])
display(fresh_df)

## 8. Comparison by query (side-by-side)

Volume file (query, region) labels are normalized to match profile canonical names via LLM, so the merge fills the `total` column. If `OPENROUTER_API_KEY` is not set, normalization is skipped and `total` may be NaN.

In [ ]:
def normalize_volume_query_region_with_llm(vol_df: pd.DataFrame, canonical_pairs: list, api_key: str) -> pd.DataFrame:
    """Use LLM to map volume file (query, region) to canonical (query, region) from profiles."""
    log = logging.getLogger("provider_eval")
    if vol_df.empty or not canonical_pairs or not api_key:
        log.info("Normalize volume: skip (empty=%s, canon_len=%s, api_key=%s)", vol_df.empty, len(canonical_pairs) if canonical_pairs else 0, bool(api_key))
        return vol_df
    log.info("Normalize volume: rows_before=%d, canonical_pairs=%d", len(vol_df), len(canonical_pairs))
    canonical_list = [{"query": str(q), "region": str(r)} for q, r in canonical_pairs]
    volume_rows = vol_df[["query", "region", "total"]].fillna("").astype(str).to_dict("records")
    if not volume_rows:
        return vol_df
    prompt = f"""We use these exact (query, region) pairs in our dataset:
{json.dumps(canonical_list, indent=2)}

The volume file has rows with possibly different labels (e.g. "Software Engineer React" vs "SWE_React", "A" vs "LATAM"):
{json.dumps(volume_rows[:50], indent=2)}

For each volume row, choose the single canonical (query, region) it refers to. Return only a JSON array of objects, each with keys: "volume_query", "volume_region", "canonical_query", "canonical_region". Use the exact canonical strings from the first list. One entry per volume row."""
    content = openrouter_chat([SYSTEM_JSON, {"role": "user", "content": prompt}], api_key=api_key)
    try:
        mapping_list = parse_llm_json(content, expect_array=True)
    except Exception as e:
        log.warning("Normalize volume: LLM parse failed: %s, response preview=%s", e, (content[:300] + "...") if len(content) > 300 else content)
        print("LLM volume mapping parse failed:", e)
        return vol_df
    key_to_canon = {}
    for m in mapping_list:
        vq, vr = str(m.get("volume_query", "")).strip(), str(m.get("volume_region", "")).strip()
        cq, cr = m.get("canonical_query", ""), m.get("canonical_region", "")
        if cq and cr:
            key_to_canon[(vq, vr)] = (cq, cr)
    log.info("Normalize volume: LLM returned %d mappings. Keys sample: %s", len(key_to_canon), list(key_to_canon.items())[:5])
    if not key_to_canon:
        return vol_df
    def apply_map(row):
        k = (str(row.get("query", "")).strip(), str(row.get("region", "")).strip())
        if k in key_to_canon:
            cq, cr = key_to_canon[k]
            row["query"] = cq
            row["region"] = cr
        return row
    return vol_df.apply(apply_map, axis=1)

def comparison_by_query(data: dict) -> pd.DataFrame:
    rows = []
    for prov, d in data.items():
        for p in d["profiles"]:
            q = get_by_path(p, d["mapping"].get("query", "")) or p.get("_query", "")
            r = get_by_path(p, d["mapping"].get("region", "")) or p.get("_region", "")
            rows.append({"provider": prov, "query": q, "region": r})
    if not rows:
        return pd.DataFrame()
    agg = pd.DataFrame(rows).groupby(["provider", "query", "region"]).size().reset_index(name="sample_count")
    vols = []
    for prov, d in data.items():
        v = d["volumes"].copy()
        if v.empty:
            continue
        canon = agg[agg["provider"] == prov][["query", "region"]].drop_duplicates()
        canonical_pairs = list(canon.itertuples(index=False, name=None))
        log = logging.getLogger("provider_eval")
        log.info("Comparison: normalizing volumes for provider %s, canonical_pairs=%d", prov, len(canonical_pairs))
        v = normalize_volume_query_region_with_llm(v, canonical_pairs, OPENROUTER_API_KEY or "")
        v["provider"] = prov
        vols.append(v)
    if vols:
        vol_df = pd.concat(vols, ignore_index=True)
        vol_df = vol_df.groupby(["provider", "query", "region"], as_index=False)["total"].first()
        agg = agg.merge(vol_df, on=["provider", "query", "region"], how="left")
    return agg

comp_df = comparison_by_query(data)
log = logging.getLogger("provider_eval")
log.info("Comparison: comp_df shape=%s, total non-null=%d", comp_df.shape, comp_df["total"].notna().sum() if "total" in comp_df.columns else 0)
display(comp_df.head(20))

## 9. Scoring and ranking

In [ ]:
def compute_ranking(data: dict, fill_key_df: pd.DataFrame, fresh_df: pd.DataFrame,
                   w_fill: float, w_fresh: float, w_vol: float) -> tuple:
    """Return (overall_ranking_df, rank_by_query_df)."""
    providers = list(data.keys())
    if not providers:
        return pd.DataFrame(), pd.DataFrame()
    key_cols = [c for c in KEY_FIELDS if c in fill_key_df.columns]
    fill_avg = (fill_key_df.set_index("provider")[key_cols].mean(axis=1) / 100.0) if key_cols else pd.Series(0.0, index=providers)
    med_days = fresh_df.set_index("provider")["median_days_ago"]
    max_days = med_days.max() or 1
    fresh_score = 1 - (med_days / max_days).fillna(0)
    totals = {}
    for prov, d in data.items():
        v = d["volumes"]
        totals[prov] = v["total"].sum() if not v.empty and "total" in v.columns else 0
    vol_series = pd.Series(totals)
    max_vol = vol_series.max() or 1
    vol_score = (vol_series / max_vol).fillna(0)
    f = fill_avg.reindex(providers).fillna(0)
    fr = fresh_score.reindex(providers).fillna(0)
    vo = vol_score.reindex(providers).fillna(0)
    score = w_fill * f + w_fresh * fr + w_vol * vo
    overall = pd.DataFrame({"provider": providers, "score": score.values}).sort_values("score", ascending=False)
    overall["rank"] = range(1, len(overall) + 1)
    return overall, pd.DataFrame()

overall_rank, rank_by_query = compute_ranking(data, fill_key, fresh_df, FILL_RATE_WEIGHT, FRESHNESS_WEIGHT, VOLUME_WEIGHT)
log = logging.getLogger("provider_eval")
log.info("Ranking: top=%s", overall_rank.head(10).to_dict("records") if not overall_rank.empty else [])
print("Overall ranking:")
display(overall_rank)

## 10. Export CSV

In [ ]:
out_dir = ROOT_DATA / "output"
out_dir.mkdir(parents=True, exist_ok=True)
fill_key.to_csv(out_dir / "fill_rate_key.csv", index=False)
if not fill_add.empty:
    fill_add.to_csv(out_dir / "fill_rate_additional.csv", index=False)
fresh_df.to_csv(out_dir / "freshness.csv", index=False)
comp_df.to_csv(out_dir / "comparison_by_query.csv", index=False)
overall_rank.to_csv(out_dir / "overall_ranking.csv", index=False)
exported = ["fill_rate_key.csv", "freshness.csv", "comparison_by_query.csv", "overall_ranking.csv"]
if not fill_add.empty:
    exported.append("fill_rate_additional.csv")
print("Exported to", out_dir)
log = logging.getLogger("provider_eval")
log.info("Export: out_dir=%s, files=%s", out_dir, exported)

## 11. OpenRouter summary (recommendation)

In [ ]:
context = f"""Providers: {list(data.keys())}.
Fill rate (key fields): {fill_key.to_dict('records')}.
Freshness: {fresh_df.to_dict('records')}.
Overall ranking: {overall_rank.to_dict('records')}.
Compare and recommend which provider to choose for buying people data. 2-3 short paragraphs: who leads, by which metrics; one clear recommendation with reasons. Write in English.
"""
log = logging.getLogger("provider_eval")
if OPENROUTER_API_KEY:
    summary = openrouter_chat([{"role": "user", "content": context}])
    print(summary)
    (out_dir / "recommendation.txt").write_text(summary, encoding="utf-8")
    log.info("Recommendation: saved to %s", out_dir / "recommendation.txt")
else:
    print("Set OPENROUTER_API_KEY to get LLM recommendation.")
    log.info("Recommendation: skipped (no API key)")
log.info("Run finished. Full log saved to: %s", LOG_FILE)
log_content = "\n".join(LOG_LINES)
LOG_FILE.write_text(log_content, encoding="utf-8")
print("Full run log:", LOG_FILE)
print("\n--- Full log content (saved in this cell output) ---\n")
print(log_content)

## 12. Dashboard — final report

In [ ]:
from IPython.display import HTML
from datetime import date
import math

def _safe(v):
    try:
        f = float(v)
        return None if math.isnan(f) else f
    except Exception:
        return None

def _bar(v, max_v=100, width=120):
    s = _safe(v)
    if s is None:
        return '<span style="color:#6b7280;font-size:12px">—</span>'
    pct = min(100, s / (max_v or 1) * 100)
    c = "#22c55e" if pct >= 70 else ("#f59e0b" if pct >= 40 else "#ef4444")
    return (
        f'<div style="display:flex;align-items:center;gap:8px">'
        f'<div style="width:{width}px;background:#1e293b;border-radius:3px;height:8px;flex-shrink:0">'
        f'<div style="width:{pct:.1f}%;height:8px;border-radius:3px;background:{c}"></div></div>'
        f'<span style="font-size:12px;color:#cbd5e1;min-width:38px">{s:.1f}%</span></div>'
    )

def build_dashboard(overall_rank, fill_key, fresh_df, comp_df, data, KEY_FIELDS,
                    FILL_RATE_WEIGHT, FRESHNESS_WEIGHT, VOLUME_WEIGHT, summary_text=""):
    providers = list(overall_rank["provider"]) if not overall_rank.empty else list(data.keys())
    PALETTE = ["#3b82f6","#f97316","#22c55e","#a855f7","#ec4899"]
    MEDALS  = {1:"🥇",2:"🥈",3:"🥉"}
    key_cols = [c for c in KEY_FIELDS if c in fill_key.columns]

    # ── Ranking cards ──────────────────────────────────────────────────────
    rank_cards = ""
    for _, row in overall_rank.iterrows():
        p = row["provider"]; rk = int(row["rank"]); sc = float(row["score"])
        col = PALETTE[(rk-1) % len(PALETTE)]
        medal = MEDALS.get(rk, f"#{rk}")
        rank_cards += f"""
        <div style="background:#1e293b;border-top:3px solid {col};border-radius:10px;padding:20px 22px;flex:1;min-width:180px">
          <div style="font-size:28px">{medal}</div>
          <div style="font-size:17px;font-weight:700;color:{col};margin:6px 0 2px">{p}</div>
          <div style="font-size:12px;color:#64748b;margin-bottom:14px">Rank #{rk}</div>
          <div style="font-size:26px;font-weight:800;color:#f1f5f9">{sc:.3f}</div>
          <div style="font-size:11px;color:#64748b;margin-top:2px">composite score</div>
        </div>"""

    # ── Score breakdown table ──────────────────────────────────────────────
    score_rows = ""
    def _vol_sum(vols):
        if vols is None or vols.empty or "total" not in vols.columns:
            return 0.0
        s = pd.to_numeric(vols["total"], errors="coerce").sum()
        return float(s) if not math.isnan(float(s)) else 0.0
    max_vol = max(_vol_sum(data[p]["volumes"]) for p in providers) or 1
    max_days = fresh_df["median_days_ago"].max() or 1

    for _, row in overall_rank.iterrows():
        p = row["provider"]
        pf = fill_key[fill_key["provider"] == p]
        fill_v = _safe(pf[key_cols].mean(axis=1).iloc[0]) if (not pf.empty and key_cols) else None
        vols = data[p]["volumes"]
        vol_total = _vol_sum(vols)
        vol_v = vol_total / max_vol * 100
        fr = fresh_df[fresh_df["provider"] == p]
        days = _safe(fr["median_days_ago"].iloc[0]) if not fr.empty else None
        fresh_v = (1 - days / max_days) * 100 if days is not None else None
        sc = float(row["score"])
        score_rows += f"""
        <tr>
          <td style="padding:10px 14px;font-weight:600;color:#f1f5f9;font-size:13px;border-bottom:1px solid #0f172a">{p}</td>
          <td style="padding:10px 14px;border-bottom:1px solid #0f172a">{_bar(fill_v)}</td>
          <td style="padding:10px 14px;border-bottom:1px solid #0f172a">{_bar(fresh_v)}</td>
          <td style="padding:10px 14px;border-bottom:1px solid #0f172a">{_bar(vol_v)}</td>
          <td style="padding:10px 14px;font-size:16px;font-weight:800;color:#f1f5f9;border-bottom:1px solid #0f172a">{sc:.3f}</td>
        </tr>"""

    # ── Fill rate table ────────────────────────────────────────────────────
    prov_ths = "".join(
        f'<th style="padding:9px 14px;color:#94a3b8;font-weight:600;font-size:12px;text-align:left">{p}</th>'
        for p in providers)
    fill_rows = ""
    for field in key_cols:
        fill_rows += f'<tr><td style="padding:8px 14px;color:#94a3b8;font-size:12px;border-bottom:1px solid #0f172a;white-space:nowrap">{field}</td>'
        for p in providers:
            pf = fill_key[fill_key["provider"] == p]
            val = _safe(pf[field].iloc[0]) if (not pf.empty and field in pf.columns) else None
            fill_rows += f'<td style="padding:8px 14px;border-bottom:1px solid #0f172a">{_bar(val, width=100)}</td>'
        fill_rows += "</tr>"

    # ── Freshness cards ────────────────────────────────────────────────────
    fresh_cards = ""
    for p in providers:
        fr = fresh_df[fresh_df["provider"] == p]
        days = _safe(fr["median_days_ago"].iloc[0]) if not fr.empty else None
        stale = _safe(fr["pct_stale_over_1y"].iloc[0]) if not fr.empty else None
        days_s  = f"{int(days)}d" if days is not None else "—"
        stale_s = f"{stale*100:.1f}%" if stale is not None else "—"
        fc = "#22c55e" if (days and days < 180) else ("#f59e0b" if days else "#64748b")
        fresh_cards += f"""
        <div style="background:#1e293b;border-radius:10px;padding:18px 22px;flex:1;min-width:160px">
          <div style="font-size:14px;font-weight:700;color:#f1f5f9;margin-bottom:14px">{p}</div>
          <div style="margin-bottom:10px">
            <div style="font-size:11px;color:#64748b;margin-bottom:2px">Median profile age</div>
            <div style="font-size:24px;font-weight:800;color:{fc}">{days_s}</div>
          </div>
          <div>
            <div style="font-size:11px;color:#64748b;margin-bottom:2px">Stale (&gt;1 year)</div>
            <div style="font-size:18px;font-weight:700;color:#f1f5f9">{stale_s}</div>
          </div>
        </div>"""

    # ── Volume cards ───────────────────────────────────────────────────────
    vol_cards = ""
    for p in providers:
        vols = data[p]["volumes"]
        total = int(_vol_sum(vols))
        pairs = len(vols) if not vols.empty else 0
        profiles = len(data[p]["profiles"])
        vol_cards += f"""
        <div style="background:#1e293b;border-radius:10px;padding:18px 22px;flex:1;min-width:160px">
          <div style="font-size:14px;font-weight:700;color:#f1f5f9;margin-bottom:14px">{p}</div>
          <div style="margin-bottom:10px">
            <div style="font-size:11px;color:#64748b;margin-bottom:2px">Market volume (total)</div>
            <div style="font-size:22px;font-weight:800;color:#3b82f6">{total:,}</div>
          </div>
          <div style="display:flex;gap:20px">
            <div>
              <div style="font-size:11px;color:#64748b;margin-bottom:2px">Queries covered</div>
              <div style="font-size:16px;font-weight:700;color:#f1f5f9">{pairs}</div>
            </div>
            <div>
              <div style="font-size:11px;color:#64748b;margin-bottom:2px">Sample profiles</div>
              <div style="font-size:16px;font-weight:700;color:#f1f5f9">{profiles:,}</div>
            </div>
          </div>
        </div>"""

    # ── Volume comparison table (top 10 queries) ───────────────────────────
    vol_cmp = ""
    if not comp_df.empty and "provider" in comp_df.columns:
        # Show each provider's top queries separately (query names differ per provider)
        provider_blocks = ""
        for pi, prov in enumerate(providers):
            prov_df = comp_df[comp_df["provider"] == prov].copy()
            if prov_df.empty:
                continue
            prov_df["total"] = pd.to_numeric(prov_df["total"], errors="coerce")
            prov_df = prov_df.dropna(subset=["total"])
            prov_df = prov_df.sort_values("total", ascending=False).head(10)
            if prov_df.empty:
                continue
            col = PALETTE[pi % len(PALETTE)]
            max_v = float(prov_df["total"].max()) or 1.0
            rows_html = ""
            for _, r in prov_df.iterrows():
                q   = str(r.get("query", ""))
                reg = str(r.get("region", ""))
                tot = _safe(r.get("total"))
                sc  = _safe(r.get("sample_count"))
                bar = _bar(tot / max_v * 100, width=120) if tot is not None else "—"
                num = f'{int(tot):,}' if tot is not None else "—"
                sc_str = f'<span style="color:#475569;font-size:11px">({int(sc)} samples)</span>' if sc is not None else ""
                rows_html += (
                    f'<tr>'
                    f'<td style="padding:8px 14px;color:#cbd5e1;font-size:12px;border-bottom:1px solid #0f172a;white-space:nowrap">{q}</td>'
                    f'<td style="padding:8px 14px;color:#64748b;font-size:12px;border-bottom:1px solid #0f172a;white-space:nowrap">{reg}</td>'
                    f'<td style="padding:8px 14px;border-bottom:1px solid #0f172a">{bar}</td>'
                    f'<td style="padding:8px 14px;font-size:12px;color:#94a3b8;border-bottom:1px solid #0f172a;text-align:right">{num} {sc_str}</td>'
                    f'</tr>'
                )
            provider_blocks += f"""
            <div style="flex:1;min-width:320px">
              <div style="font-size:12px;font-weight:700;color:{col};margin-bottom:10px;padding-bottom:6px;border-bottom:1px solid {col}33">{prov}</div>
              <table style="width:100%;border-collapse:collapse;background:#1e293b;border-radius:8px;overflow:hidden">
                <tr style="background:#0f172a">
                  <th style="padding:7px 14px;color:#64748b;font-size:11px;font-weight:600;text-align:left">Query</th>
                  <th style="padding:7px 14px;color:#64748b;font-size:11px;font-weight:600;text-align:left">Region</th>
                  <th style="padding:7px 14px;color:#64748b;font-size:11px;font-weight:600;text-align:left">Volume</th>
                  <th style="padding:7px 14px;color:#64748b;font-size:11px;font-weight:600;text-align:right">Count</th>
                </tr>
                {rows_html}
              </table>
            </div>"""
        vol_cmp = f"""
        <div class="pe-section">
          <div class="pe-label">Volume by Query / Region (top 10 per provider)</div>
          <div style="display:flex;gap:20px;flex-wrap:wrap;align-items:flex-start">{provider_blocks}</div>
        </div>"""

    # ── Recommendation ─────────────────────────────────────────────────────
    if summary_text:
        paras = [p.strip() for p in summary_text.split("\n") if p.strip()]
        rec_body = "".join(f'<p style="margin:0 0 12px 0;line-height:1.75;color:#cbd5e1;font-size:14px">{p}</p>' for p in paras)
    else:
        rec_body = '<p style="color:#64748b;margin:0;font-size:13px">Set OPENROUTER_API_KEY to get an AI recommendation.</p>'

    html = f"""
<style>
  .pe-dashboard*{{box-sizing:border-box;font-family:"Inter","Segoe UI",system-ui,sans-serif}}
  .pe-section{{margin-bottom:32px}}
  .pe-label{{font-size:11px;font-weight:700;letter-spacing:2px;text-transform:uppercase;
             color:#3b82f6;margin-bottom:14px;padding-bottom:8px;border-bottom:1px solid #1e293b}}
  .pe-row{{display:flex;gap:14px;flex-wrap:wrap}}
</style>
<div class="pe-dashboard" style="background:#0f172a;border-radius:14px;padding:36px;
     color:#f1f5f9;max-width:1080px;margin:0 auto;box-shadow:0 4px 40px #0004">

  <!-- Header -->
  <div style="margin-bottom:32px;padding-bottom:20px;border-bottom:1px solid #1e293b;
       display:flex;align-items:baseline;justify-content:space-between;flex-wrap:wrap;gap:8px">
    <div>
      <div style="font-size:20px;font-weight:800;color:#f1f5f9;letter-spacing:-0.3px">
        Provider Evaluation Report</div>
      <div style="font-size:12px;color:#64748b;margin-top:4px">
        {date.today().strftime("%B %d, %Y")} &nbsp;·&nbsp;
        {len(providers)} provider(s) &nbsp;·&nbsp;
        weights: fill {FILL_RATE_WEIGHT:.0%} / freshness {FRESHNESS_WEIGHT:.0%} / volume {VOLUME_WEIGHT:.0%}
      </div>
      <button class="pe-print-btn" onclick="window.print()">🖨 Save as PDF</button>
    </div>
  </div>

  <!-- Ranking -->
  <div class="pe-section">
    <div class="pe-label">Overall Ranking</div>
    <div class="pe-row">{rank_cards}</div>
  </div>

  <!-- Score breakdown -->
  <div class="pe-section">
    <div class="pe-label">Score Breakdown</div>
    <div style="overflow-x:auto">
    <table style="width:100%;border-collapse:collapse;background:#1e293b;border-radius:10px;overflow:hidden">
      <tr style="background:#0f172a">
        <th style="padding:10px 14px;color:#94a3b8;font-weight:600;font-size:12px;text-align:left">Provider</th>
        <th style="padding:10px 14px;color:#94a3b8;font-weight:600;font-size:12px;text-align:left">Fill rate ({FILL_RATE_WEIGHT:.0%})</th>
        <th style="padding:10px 14px;color:#94a3b8;font-weight:600;font-size:12px;text-align:left">Freshness ({FRESHNESS_WEIGHT:.0%})</th>
        <th style="padding:10px 14px;color:#94a3b8;font-weight:600;font-size:12px;text-align:left">Volume ({VOLUME_WEIGHT:.0%})</th>
        <th style="padding:10px 14px;color:#94a3b8;font-weight:600;font-size:12px;text-align:left">Score</th>
      </tr>
      {score_rows}
    </table></div>
  </div>

  <!-- Key field fill rate -->
  <div class="pe-section">
    <div class="pe-label">Key Field Fill Rate</div>
    <div style="overflow-x:auto">
    <table style="width:100%;border-collapse:collapse;background:#1e293b;border-radius:10px;overflow:hidden">
      <tr style="background:#0f172a">
        <th style="padding:9px 14px;color:#94a3b8;font-weight:600;font-size:12px;text-align:left">Field</th>
        {prov_ths}
      </tr>
      {fill_rows}
    </table></div>
  </div>

  <!-- Freshness + Volume -->
  <div class="pe-section">
    <div class="pe-label">Data Freshness</div>
    <div class="pe-row">{fresh_cards}</div>
  </div>
  <div class="pe-section">
    <div class="pe-label">Volume &amp; Coverage</div>
    <div class="pe-row">{vol_cards}</div>
  </div>

  {vol_cmp}

  <!-- AI Recommendation -->
  <div class="pe-section" style="margin-bottom:0">
    <div class="pe-label">AI Recommendation</div>
    <div style="background:#1e293b;border-left:3px solid #3b82f6;border-radius:10px;padding:22px 26px">
      {rec_body}
    </div>
  </div>

</div>
"""
    return html

_summary_text = summary if "summary" in dir() and isinstance(summary, str) else ""
display(HTML(build_dashboard(
    overall_rank, fill_key, fresh_df, comp_df, data, KEY_FIELDS,
    FILL_RATE_WEIGHT, FRESHNESS_WEIGHT, VOLUME_WEIGHT,
    summary_text=_summary_text,
)))


## 13. Export report (HTML + PDF)

In [ ]:
# Export report: save standalone HTML + try WeasyPrint PDF
import sys, subprocess

_dashboard_html = build_dashboard(
    overall_rank, fill_key, fresh_df, comp_df, data, KEY_FIELDS,
    FILL_RATE_WEIGHT, FRESHNESS_WEIGHT, VOLUME_WEIGHT,
    summary_text=(_summary_text if "_summary_text" in dir() else ""),
)

_full_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Provider Evaluation Report</title>
  <style>
    body {{ margin: 0; padding: 24px; background: #060d1a; }}
    @media print {{
      @page {{ size: A4; margin: 12mm 10mm; }}
      body {{ background: #0f172a; -webkit-print-color-adjust: exact; print-color-adjust: exact; }}
      .pe-print-btn {{ display: none !important; }}
    }}
  </style>
</head>
<body>{_dashboard_html}</body>
</html>"""

# Save HTML
_html_path = out_dir / "provider_report.html"
_html_path.write_text(_full_html, encoding="utf-8")
print("HTML report saved:", _html_path)

# Try WeasyPrint PDF
_pdf_path = out_dir / "provider_report.pdf"
try:
    try:
        from weasyprint import HTML as _WP
    except ImportError:
        print("Installing WeasyPrint...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "weasyprint"], stdout=subprocess.DEVNULL)
        from weasyprint import HTML as _WP
    _WP(string=_full_html, base_url=str(out_dir)).write_pdf(str(_pdf_path))
    print("PDF report saved:", _pdf_path)
    _pdf_ok = True
except Exception as _e:
    print(f"WeasyPrint PDF failed ({_e}). Use the HTML file: open in browser → Ctrl+P → Save as PDF.")
    _pdf_ok = False

# Colab: download files
if IN_COLAB:
    from google.colab import files
    files.download(str(_html_path))
    if _pdf_ok:
        files.download(str(_pdf_path))
else:
    print("Files saved to:", out_dir)


In [ ]:
# Show log file location and content (log lives on Colab VM at /content/, not on your local disk)
print("ROOT_DATA (log directory):", ROOT_DATA)
print("Log file path:", LOG_FILE)
print("Exists:", LOG_FILE.exists())
if LOG_FILE.exists():
    print("\n--- Log content ---")
    print(LOG_FILE.read_text(encoding="utf-8"))
    if IN_COLAB:
        from google.colab import files
        files.download(str(LOG_FILE))  # download to your computer
else:
    print("Log file not found (e.g. runtime was restarted and /content was cleared).")